In [1]:
import pandas as pd
import re

# 1. LOAD DATA
df = pd.read_csv("post_pandemic_health_analysis.csv")
# Clean column names
df.columns = df.columns.str.strip()
# Fill missing values
df = df.fillna("Unknown")
# Drop unnecessary columns safely
df = df.drop(columns=['Timestamp'], errors='ignore')
df = df.drop(columns=[' '], errors='ignore')

# 2. CLEAN GENDER
df['Gender'] = df['Gender'].str.strip().str.lower()
df['Gender'] = df['Gender'].replace({
    'm': 'male',
    'f': 'female'
})

# 3. CREATE CATEGORY (AGE + GENDER)
def categorize(row):
    age = str(row['Your Age group during pandemic']).lower()
    gender = str(row['Gender']).lower()
    
    if gender == 'male':
        g = 'Male'
        teen_g = 'Boy'
    elif gender == 'female':
        g = 'Female'
        teen_g = 'Girl'
    else:
        g = 'Unknown'
        teen_g = 'Teen'
    
    if 'elder' in age:
        return f"Elderly {g}"
    elif 'middle' in age:
        return f"Middle Age {g}"
    elif 'teen' in age:
        return f"Teen {teen_g}"
    else:
        return f"Adult {g}"

df['Category'] = df.apply(categorize, axis=1)


In [2]:
import pandas as pd
import re

# 1. COPY DATA
df_clean = df.copy()

# 2. NORMALIZE FUNCTION
def normalize(s):
    s = str(s).lower().strip()

    # fix repeated words (pain pain → pain)
    s = re.sub(r'\b(\w+)\s+\1\b', r'\1', s)

    # fix spelling mistake
    s = s.replace("cholestrol", "cholesterol")

    return s

# 3. STANDARDIZATION
def standardize(s):
    s = normalize(s)

    if "joint pain" in s:
        return "joint pain"
    if "shoulder pain" in s:
        return "shoulder pain"
    if "neck pain" in s:
        return "neck pain"
    if "back pain" in s:
        return "back pain"
    if "leg pain" in s:
        return "leg pain"

    if "fatigue" in s:
        return "fatigue"
    if "stress" in s:
        return "stress"
    if "anxiety" in s:
        return "anxiety"
    if "sleep" in s:
        return "sleep issue"

    if "hormonal" in s:
        return "hormonal imbalance"
    if "menstrual" in s:
        return "menstrual issue"

    if "glucose" in s:
        return "glucose"
    if "cholesterol" in s:
        return "cholesterol"

    if "no symptoms" in s:
        return "no symptoms"

    return s

# 4. PROCESS FUNCTION
def process(x):
    x = str(x)
    symptoms = [i.strip() for i in x.split(';') if i.strip()]
    cleaned = [standardize(s) for s in symptoms]
    return list(set(cleaned))   # remove duplicates per person

# 5. APPLY CLEANING
df_clean['Symptoms_processed'] = df_clean['Symptoms'].apply(process)

# 6. EXPLODE (IMPORTANT FOR ANALYSIS)
df_exploded = df_clean.explode('Symptoms_processed')

# 7. ANALYSIS
print(df_exploded['Symptoms_processed'].value_counts())

# 8. SAVE CLEAN DATA
df_clean.to_csv("cleaned_data.csv", index=False)

Symptoms_processed
fatigue                       37
stress                        19
joint pain                    18
no symptoms                   14
hormonal imbalance            12
menstrual issue               11
leg pain                      10
shoulder pain                 10
glucose                        9
cholesterol                    8
back pain                      7
decreased strength             7
low strength                   7
sleep issue                    5
sprain                         5
anxiety                        5
cramps                         5
infertility                    4
neck pain                      2
sprain pain                    1
thigh and calf muscle pain     1
Name: count, dtype: int64
